# Analyze one run

Front end to the `hydrogen_surface_kinetics` package. It runs the same analyses as `bash run.sh runs/<name>` and shares its cache, so anything already solved there loads here in seconds (and the other way round).

1. Set `SETTINGS` to the run's `settings.py`.
2. Run the cells from the top. Each analysis cell can change that analysis' settings first. A change creates new cache entries and leaves the old ones alone.
3. Tables are in `ctx.outputs[...]` (pandas DataFrames). CSVs and figures are saved in the run's `output/` folder.

Long batches (the full sweep, rate control) are better run from a terminal with `bash run.sh`, which keeps the Mac awake.

Kernel: **rmg4.0** (only that environment imports the compiled RMG-Py, which `flux` needs).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

import hydrogen_surface_kinetics as hsk
from hydrogen_surface_kinetics import analyses

pd.set_option('display.width', 250, 'display.max_columns', 30, 'display.max_colwidth', 60)

SETTINGS = '../runs/ni_first_try/settings.py'   # the run to analyze
RUN = dict(workers=None, recompute=None)     # e.g. workers=4, recompute=True


def show(*names, width=900):
    """Display saved figures from the run's output folder."""
    for name in names:
        display(Image(filename=ctx.path(name), width=width))

## 1. Input and mechanism
`load_config` reads `settings.py`. Keyword arguments override it, and dictionary sections merge key by key, e.g. `hsk.load_config(SETTINGS, temperatures=[300, 500], curves={'ratios': [1e-4, 1e-2]})`.

In [ ]:
cfg = hsk.load_config(SETTINGS)
ctx = analyses.run(cfg, only=[], **RUN)          # loads the model; runs nothing yet
print('\nanalyses in settings.py:', cfg.analyses)
print('available:', hsk.config.ANALYSIS_ORDER)

In [ ]:
from hydrogen_surface_kinetics.conditions import main_conditions
main_conditions(ctx.model).drop(columns='X')

## 2. Main steady states
Forward dissociation rate, coverages and poisoning factor for every case and temperature, from two starting surfaces.

In [ ]:
analyses.run(ctx=ctx, only='steady_state')
ss = ctx.outputs['steady_state']
ss['status'].drop(columns='key')

In [ ]:
cols = ['basis', 'T_K', 'case', 'tof_fwd', 'tof_net', 'theta_free', 'theta_probe', 'theta_molec',
        'site_count_factor', 'poisoning_factor', 'slowdown_x', 'starts_agree']
display(ss['summary'][[c for c in cols if c in ss['summary']]])
show('rates_summary.png')

## 3. What covers the surface

In [ ]:
ctx.cfg.composition.update(min_listed=1e-3)    # list species above this coverage
analyses.run(ctx=ctx, only='composition')
show('surface_composition.png')
ctx.outputs['composition']['on_surface']

## 4. Mechanism diagnostics
Binding enthalpies vs literature, a what-if with the molecular adsorbate's reactions switched off, and the main sources and sinks of chosen species.

In [ ]:
analyses.run(ctx=ctx, only='diagnostics')
d = ctx.outputs['diagnostics']
for name in ('binding', 'switch_off', 'top_reactions'):
    if name in d:
        display(d[name])

## 5. Poisoning curves (rate vs impurity level)

In [ ]:
# ctx.cfg.curves.update(ratios=[1e-6, 1e-4, 1e-2, 0.25], temperatures=[300.0])
analyses.run(ctx=ctx, only='curves')
show('poisoning_curves.png')
ctx.outputs['curves']['tolerance']

## 6. Temperature sweep

In [ ]:
# ctx.cfg.sweep.update(temperatures=[300.0, 400.0, 500.0])
analyses.run(ctx=ctx, only='sweep')
show('temperature_sweep.png')
ctx.outputs['sweep']['release']

## 7. Reaction orders

In [ ]:
analyses.run(ctx=ctx, only='orders')
ctx.outputs['orders']['orders']

## 8. Degree of rate control (DRC) and thermodynamic rate control (TRC)
The largest batch: about 50 steady states per case.

In [ ]:
# ctx.cfg.rate_control.update(cases=['H2 only', 'CO'], drc_top_n=10)
analyses.run(ctx=ctx, only='rate_control')
sens = ctx.outputs['rate_control']['sensitivity']
for (kind, T, case), part in sens.groupby(['kind', 'T_K', 'case'], sort=False):
    part = part.reindex(part['X_tof_fwd'].abs().sort_values(ascending=False).index)
    print(f'{kind}, {case}, {T:.0f} K   (sum of X = {part["X_tof_fwd"].sum():.3f})')
    display(part[['X_tof_fwd', 'X_probe_consumed', 'label']].head(6))

## 9. Reaction path diagrams

In [ ]:
analyses.run(ctx=ctx, only='paths')
for files in ctx.outputs['paths'].values():
    png = [f for f in files if f.endswith('.png')]
    if png:
        display(Image(filename=png[0], width=700))

## 10. Poisoning over time

In [ ]:
analyses.run(ctx=ctx, only='transients')
show('poisoning_over_time.png')
ctx.outputs['transients']['summary']

## 11. Flux-diagram videos (optional, needs RMG-Py)
Runs the RMG input file's own batch reactors and writes a video for each into `output/flux/<n>/`. It takes about 2 minutes per reactor at `n_points = 12`, and longer at the default 100.

In [ ]:
RUN_FLUX = False
if RUN_FLUX:
    ctx.cfg.flux.update(reactor_indices=[24], n_points=45)
    analyses.run(ctx=ctx, only='flux')